In [1]:
# Test data repair scripts 

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import json

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

sys.path.append(os.path.join(ROOT_PATH, "agent/scripts/ca"))

from data_repair_ca_avenal import data_repair
MY_JURISDICTION = "Avenal"

INPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data", "permits_ca_sample.parquet")


In [2]:
df = pd.read_parquet(INPUT_FILEPATH)
sub_df = df[df["JURISDICTION"] == MY_JURISDICTION]

for col in ['FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE']:
    sub_df[f'{col}_FLAG'] = ""

#sub_df_filled = sub_df.copy()
sub_df_filled = data_repair(sub_df)

assert(len(sub_df) == len(sub_df_filled))


In [3]:
print(f"FILE_DATE available (all): {sub_df['FILE_DATE'].notna().mean():.1%} -> {sub_df_filled['FILE_DATE'].notna().mean():.1%}")

print(f"PERMIT_DATE available (all): {sub_df['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled['PERMIT_DATE'].notna().mean():.1%}")

print(f"FINAL_DATE available (all): {sub_df['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled['FINAL_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Active', 'Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Active', 'Final'])
print(f"PERMIT_DATE available (active/final): {sub_df.loc[mask1]['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['PERMIT_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Final'])
print(f"FINAL_DATE available (final): {sub_df.loc[mask1]['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['FINAL_DATE'].notna().mean():.1%}")


FILE_DATE available (all): 100.0% -> 100.0%
PERMIT_DATE available (all): 0.0% -> 0.0%
FINAL_DATE available (all): 0.0% -> 0.0%
PERMIT_DATE available (active/final): 0.0% -> 0.0%
FINAL_DATE available (final): 0.0% -> 0.0%


In [4]:
for col in ['STATUS_NORMALIZED', 'FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE']:
    print(sub_df_filled[f'{col}_FLAG'].value_counts())

STATUS_NORMALIZED_FLAG
FIXED    49
Name: count, dtype: int64
FILE_DATE_FLAG
FIXED    209
Name: count, dtype: int64
Series([], Name: count, dtype: int64)
Series([], Name: count, dtype: int64)


In [5]:
print(sub_df['STATUS_NORMALIZED'].value_counts())
print(sub_df_filled['STATUS_NORMALIZED'].value_counts())

STATUS_NORMALIZED
Final        835
Active       590
In Review    533
Inactive      42
Name: count, dtype: int64
STATUS_NORMALIZED
Final        856
Active       565
In Review    536
Inactive      43
Name: count, dtype: int64


In [6]:
mask = sub_df_filled["FINAL_DATE"].isna()
#mask = sub_df_filled["JURISDICTION"].notna()
sample = sub_df_filled.loc[mask].sample(1).iloc[0]
DATA = sample["DATA"]
DATES_DATA = du.extract_date_fields(DATA) 

print(f"STATUS_NORMALIZED: {sample['STATUS_NORMALIZED']}    *Filled: {sample['STATUS_NORMALIZED_FLAG']}*")
print(f"RECORD_TYPE_ORIGINAL: {sample['RECORD_TYPE_ORIGINAL']}")
print(f"FILE_DATE: {sample['FILE_DATE']}       *Filled: {sample['FILE_DATE_FLAG']}*")
print(f"PERMIT_DATE: {sample['PERMIT_DATE']}   *Filled: {sample['PERMIT_DATE_FLAG']}*")
print(f"FINAL_DATE: {sample['FINAL_DATE']}     *Filled: {sample['FINAL_DATE_FLAG']}*")

print("DATES_DATA: ")
print(json.dumps(DATES_DATA, indent=2))



STATUS_NORMALIZED: Inactive    *Filled: nan*
RECORD_TYPE_ORIGINAL: Business License (Non-Fixed)
FILE_DATE: 2022-01-31       *Filled: nan*
PERMIT_DATE: None   *Filled: nan*
FINAL_DATE: None     *Filled: nan*
DATES_DATA: 
{
  "main": {
    "status": -1,
    "dateCreated": "2022-01-31T18:11:30.466Z",
    "formComplete": false,
    "dateSubmitted": "2022-01-31T18:22:24.113Z",
    "expirationDate": null,
    "lastUpdatedDate": "2022-02-02T00:28:47.000Z",
    "lastUpdatedByUserID": "auth0|605385ec68e8c40071d666bc"
  },
  "extra": {
    "Clerk Date": "",
    "Expiration Date": "",
    "Planning Dept Date": "",
    "Date of Certification": "",
    "Business Ownership Status": "Sole Proprietorship",
    "Start Date of Business in Avenal": "01/31/2022"
  }
}


In [7]:
print("DATA:")
print(json.dumps(json.loads(DATA), indent=2))



DATA:
{
  "main": {
    "id": 250,
    "mbl": null,
    "city": "Avenal",
    "unit": "111",
    "label": null,
    "state": "CA",
    "value": null,
    "water": null,
    "formID": null,
    "prefix": "NFIX-",
    "renews": false,
    "sewage": null,
    "status": -1,
    "zoning": null,
    "country": "US",
    "lotArea": null,
    "zipCode": "93204",
    "bookPage": null,
    "latitude": "35.997381",
    "recordID": 250,
    "recordNo": "NFIX-31",
    "streetNo": "1068",
    "isEnabled": true,
    "longitude": "-120.129835",
    "ownerCity": "Avenal",
    "ownerName": "Angela Torres ",
    "ownerUnit": "111",
    "projectID": null,
    "renewalNo": null,
    "yearBuilt": null,
    "locationID": 15115,
    "ownerEmail": "Imarie080@gmail.com",
    "ownerState": "CA",
    "postalCode": "93204",
    "streetName": "South 7th Avenue",
    "dateCreated": "2022-01-31T18:11:30.466Z",
    "description": null,
    "fullAddress": "1068 South 7th Avenue, Unit 111, Avenal, CA 93204",
    "proper